In [1]:
# Installation
!pip install -q transformers datasets torch scikit-learn

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup
)
from sklearn.metrics import f1_score, classification_report
import warnings
import gc
import random
warnings.filterwarnings('ignore')

# Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Device: {DEVICE}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("✅ Imports terminés!")

🔥 Device: cuda
✅ Imports terminés!


In [2]:
# Chargement
df = pd.read_csv('/content/sent_train_v2.csv')
test_df = pd.read_csv('/content/test_unlabeled.csv')

# Renommage
df = df.rename(columns={'Sentence': 'text', 'Sentiment': 'sentiment'})
test_df = test_df.rename(columns={'Sentence': 'text'})

# Mapping
sentiment_map = {'positive': 2, 'neutral': 1, 'negative': 0}
df['label'] = df['sentiment'].map(sentiment_map)

print("📊 Distribution initiale:")
print(f"Train: {len(df)}, Test: {len(test_df)}")
print(df['label'].value_counts().sort_index())

# Nettoyage
def clean_text(text):
    text = str(text).strip()
    text = ' '.join(text.split())
    return text

df['text'] = df['text'].apply(clean_text)
test_df['text'] = test_df['text'].apply(clean_text)

# ============================================================================
# 🔄 AUGMENTATION LÉGÈRE (CLEF POUR 84.61%)
# ============================================================================

import random

class LightAugmenter:
    def __init__(self):
        self.synonyms = {
            'رائع': ['ممتاز', 'جميل', 'حلو'],
            'جيد': ['كويس', 'تمام', 'منيح'],
            'ممتاز': ['رائع', 'جميل'],
            'سيء': ['خايب', 'وحش'],
            'مشكلة': ['إشكال', 'قضية'],
            'كان': ['كانت', 'صار'],
            'الفندق': ['الأوتيل', 'المكان'],
        }

    def synonym_replace(self, text, n=1):
        words = text.split()
        for word in words:
            if word in self.synonyms and random.random() > 0.5:
                synonym = random.choice(self.synonyms[word])
                text = text.replace(word, synonym, 1)
                break
        return text

    def augment(self, text):
        return self.synonym_replace(text, n=1)

augmenter = LightAugmenter()

# Équilibrage
df_by_label = {label: df[df['label'] == label].copy() for label in [0, 1, 2]}
max_samples = max(len(df_by_label[label]) for label in [0, 1, 2])

print(f"\n🔄 Augmentation légère...")
augmented_dfs = []

for label in [0, 1, 2]:
    df_label = df_by_label[label]
    current_size = len(df_label)
    needed = max_samples - current_size

    print(f"Label {label}: {current_size} → {max_samples} (+{needed})")

    if needed > 0:
        augmented_rows = []
        for i in range(needed):
            idx = i % current_size
            row = df_label.iloc[idx]
            aug_text = augmenter.augment(row['text'])

            augmented_rows.append({
                'text': aug_text,
                'label': label,
                'sentiment': row['sentiment'],
                'dialect': row['dialect']
            })

        df_augmented = pd.concat([df_label, pd.DataFrame(augmented_rows)], ignore_index=True)
    else:
        df_augmented = df_label

    augmented_dfs.append(df_augmented)

df_balanced = pd.concat(augmented_dfs, ignore_index=True)
df_balanced = df_balanced.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"\n✅ Dataset final: {len(df_balanced)} samples")
print(df_balanced['label'].value_counts().sort_index())

📊 Distribution initiale:
Train: 1731, Test: 312
label
0    657
1    465
2    609
Name: count, dtype: int64

🔄 Augmentation légère...
Label 0: 657 → 657 (+0)
Label 1: 465 → 657 (+192)
Label 2: 609 → 657 (+48)

✅ Dataset final: 1971 samples
label
0    657
1    657
2    657
Name: count, dtype: int64


In [3]:
class SimpleAttentionMARBERT(nn.Module):
    def __init__(
        self,
        model_name="UBC-NLP/MARBERT",
        num_labels=3,
        hidden_size=768,
        num_attention_heads=12,
        dropout=0.35
    ):
        super().__init__()

        self.transformer = AutoModel.from_pretrained(model_name)

        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_attention_heads,
            dropout=dropout,
            batch_first=True
        )

        self.attention_weights = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.Tanh(),
            nn.Linear(256, 1)
        )

        self.layer_norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        sequence_output = outputs.last_hidden_state

        attn_output, _ = self.attention(sequence_output, sequence_output, sequence_output)
        attn_output = self.dropout(attn_output)
        attn_output = self.layer_norm(sequence_output + attn_output)

        attention_scores = self.attention_weights(attn_output)
        attention_scores = F.softmax(attention_scores, dim=1)
        pooled_output = torch.sum(attention_scores * attn_output, dim=1)

        logits = self.classifier(pooled_output)
        return logits

print("✅ Architecture créée!")

✅ Architecture créée!


In [4]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.5, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss


class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, epsilon=0.1, weight=None):
        super().__init__()
        self.epsilon = epsilon
        self.weight = weight

    def forward(self, inputs, targets):
        n_classes = inputs.size(-1)
        log_probs = F.log_softmax(inputs, dim=-1)

        targets_one_hot = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
        targets_one_hot = targets_one_hot * (1 - self.epsilon) + self.epsilon / n_classes

        loss = (-targets_one_hot * log_probs).sum(dim=-1)

        if self.weight is not None:
            loss = loss * self.weight[targets]

        return loss.mean()


class CombinedLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.5, epsilon=0.1, focal_weight=0.7):
        super().__init__()
        self.focal = FocalLoss(alpha=alpha, gamma=gamma)
        self.label_smooth = LabelSmoothingCrossEntropy(epsilon=epsilon, weight=alpha)
        self.focal_weight = focal_weight

    def forward(self, inputs, targets):
        focal_loss = self.focal(inputs, targets)
        smooth_loss = self.label_smooth(inputs, targets)
        return self.focal_weight * focal_loss + (1 - self.focal_weight) * smooth_loss


# Class weights
labels = df_balanced['label'].values
class_counts = np.bincount(labels, minlength=3)
class_weights = 1.0 / (class_counts + 1e-6)
class_weights = class_weights / class_weights.sum() * len(class_weights)
class_weights = torch.FloatTensor(class_weights).to(DEVICE)

criterion = CombinedLoss(
    alpha=class_weights,
    gamma=2.5,
    epsilon=0.1,
    focal_weight=0.7
)

print("✅ Loss créée!")

✅ Loss créée!


In [5]:
class ArabicSentimentDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item


# ============================================================================
# TTA (Test-Time Augmentation) - CLEF POUR 84.61%
# ============================================================================

def apply_tta(texts, num_rounds=5):
    """Applique TTA sur les textes"""
    augmenter = LightAugmenter()
    all_variations = [texts]  # Original

    for _ in range(num_rounds - 1):
        variations = [augmenter.augment(text) for text in texts]
        all_variations.append(variations)

    return all_variations

print("✅ Dataset + TTA créés!")

✅ Dataset + TTA créés!


In [7]:
# ============================================================================
# 📊 STATISTIQUES DÉTAILLÉES - AVANT ET APRÈS AUGMENTATION
# ============================================================================

print("\n" + "="*80)
print("📊 STATISTIQUES COMPLÈTES DU DATASET")
print("="*80)

# ============================================================================
# AVANT AUGMENTATION
# ============================================================================
print("\n🔵 AVANT AUGMENTATION:")
print(f"   Total samples: {len(df)}")
print(f"\n   Distribution par classe:")

original_dist = df['label'].value_counts().sort_index()
label_names = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

for label in [0, 1, 2]:
    count = original_dist.get(label, 0)
    percentage = (count / len(df)) * 100
    print(f"      {label_names[label]:8} (label {label}): {count:4} samples ({percentage:5.2f}%)")

print(f"\n   Distribution par dialecte:")
for dialect, count in df['dialect'].value_counts().items():
    percentage = (count / len(df)) * 100
    print(f"      {dialect:12}: {count:4} samples ({percentage:5.2f}%)")

# ============================================================================
# APRÈS AUGMENTATION
# ============================================================================
print("\n🟢 APRÈS AUGMENTATION:")
print(f"   Total samples: {len(df_balanced)}")
print(f"\n   Distribution par classe:")

balanced_dist = df_balanced['label'].value_counts().sort_index()

for label in [0, 1, 2]:
    count = balanced_dist.get(label, 0)
    percentage = (count / len(df_balanced)) * 100
    original_count = original_dist.get(label, 0)
    added = count - original_count
    print(f"      {label_names[label]:8} (label {label}): {count:4} samples ({percentage:5.2f}%) [+{added:3} ajoutés]")

print(f"\n   Distribution par dialecte:")
for dialect, count in df_balanced['dialect'].value_counts().items():
    percentage = (count / len(df_balanced)) * 100
    original_count = df[df['dialect'] == dialect].shape[0]
    added = count - original_count
    print(f"      {dialect:12}: {count:4} samples ({percentage:5.2f}%) [+{added:3} ajoutés]")

# ============================================================================
# RÉSUMÉ DES AJOUTS
# ============================================================================
print("\n📈 RÉSUMÉ DE L'AUGMENTATION:")
total_added = len(df_balanced) - len(df)
augmentation_rate = (total_added / len(df)) * 100

print(f"   Samples originaux:  {len(df)}")
print(f"   Samples ajoutés:    {total_added}")
print(f"   Samples finaux:     {len(df_balanced)}")
print(f"   Taux d'augmentation: {augmentation_rate:.1f}%")

print(f"\n   Ajouts par classe:")
for label in [0, 1, 2]:
    original_count = original_dist.get(label, 0)
    final_count = balanced_dist.get(label, 0)
    added = final_count - original_count
    if added > 0:
        aug_rate = (added / original_count) * 100
        print(f"      {label_names[label]:8}: {added:3} samples ajoutés (+{aug_rate:.1f}%)")
    else:
        print(f"      {label_names[label]:8}: Aucun ajout (classe majoritaire)")

# ============================================================================
# EXEMPLES DE PHRASES AUGMENTÉES
# ============================================================================
print("\n📝 EXEMPLES DE PHRASES AUGMENTÉES:")
print("-" * 80)

# Trouver des exemples augmentés pour chaque classe
for label in [0, 1, 2]:
    print(f"\n{label_names[label].upper()} (label {label}):")

    # Sélectionner quelques exemples augmentés
    original_indices = set(df[df['label'] == label].index)
    augmented_samples = df_balanced[
        (df_balanced['label'] == label) &
        (~df_balanced.index.isin(original_indices))
    ].head(3)  # 3 exemples max

    if len(augmented_samples) > 0:
        for idx, (i, row) in enumerate(augmented_samples.iterrows(), 1):
            text_preview = row['text'][:80] + "..." if len(row['text']) > 80 else row['text']
            print(f"   Exemple {idx}: {text_preview}")
            print(f"              (dialecte: {row['dialect']})")
    else:
        print("   (Aucune phrase augmentée pour cette classe)")

print("\n" + "="*80)
print("✅ Analyse complète terminée!")
print("="*80 + "\n")


📊 STATISTIQUES COMPLÈTES DU DATASET

🔵 AVANT AUGMENTATION:
   Total samples: 1731

   Distribution par classe:
      Negative (label 0):  657 samples (37.95%)
      Neutral  (label 1):  465 samples (26.86%)
      Positive (label 2):  609 samples (35.18%)

   Distribution par dialecte:
      Darija      :  433 samples (25.01%)
      Saudi       :  433 samples (25.01%)
      Jordanian   :  433 samples (25.01%)
      Egyptian    :  432 samples (24.96%)

🟢 APRÈS AUGMENTATION:
   Total samples: 1971

   Distribution par classe:
      Negative (label 0):  657 samples (33.33%) [+  0 ajoutés]
      Neutral  (label 1):  657 samples (33.33%) [+192 ajoutés]
      Positive (label 2):  657 samples (33.33%) [+ 48 ajoutés]

   Distribution par dialecte:
      Darija      :  596 samples (30.24%) [+163 ajoutés]
      Saudi       :  510 samples (25.88%) [+ 77 ajoutés]
      Jordanian   :  433 samples (21.97%) [+  0 ajoutés]
      Egyptian    :  432 samples (21.92%) [+  0 ajoutés]

📈 RÉSUMÉ DE L'AUGMENT

In [6]:
from torch.optim.swa_utils import AveragedModel, SWALR

# ============================================================================
# CONFIGURATION EXACTE POUR 84.61%
# ============================================================================

MODEL_NAME = "UBC-NLP/MARBERT"
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 15  # Clé : 15 epochs
LR = 1.8e-5
ACCUMULATION_STEPS = 2

NUM_SEEDS = 7  # Clé : 7 seeds

USE_SWA = True
SWA_START_EPOCH = 10
SWA_LR = 4e-6

DROPOUT = 0.35

# TTA
USE_TTA = True
TTA_ROUNDS = 5

# Pseudo-labeling
USE_PSEUDO_LABELING = True
PSEUDO_THRESHOLD = 0.95

print("="*80)
print("🎯 CONFIGURATION POUR 84.61%")
print("="*80)
print(f"Dataset: {len(df_balanced)} samples (avec augmentation)")
print(f"Epochs: {EPOCHS}")
print(f"Seeds: {NUM_SEEDS}")
print(f"SWA: {USE_SWA} (start: {SWA_START_EPOCH})")
print(f"TTA: {USE_TTA} ({TTA_ROUNDS} rounds)")
print(f"Pseudo-labeling: {USE_PSEUDO_LABELING} (threshold: {PSEUDO_THRESHOLD})")
print("="*80)

🎯 CONFIGURATION POUR 84.61%
Dataset: 1971 samples (avec augmentation)
Epochs: 15
Seeds: 7
SWA: True (start: 10)
TTA: True (5 rounds)
Pseudo-labeling: True (threshold: 0.95)


In [9]:
# ============================================================================
# ENTRAÎNEMENT COMPLET POUR 84.61%
# ============================================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("\n" + "="*80)
print(f"🚀 ENTRAÎNEMENT ULTRA-OPTIMISÉ: {NUM_SEEDS} RUNS")
print("="*80)

all_predictions = []
all_swa_predictions = []

seeds = [42, 123, 2024, 777, 1337, 9999, 555][:NUM_SEEDS]

for run_idx, seed in enumerate(seeds):
    print(f"\n{'='*80}")
    print(f"🚀 RUN {run_idx+1}/{NUM_SEEDS} - SEED {seed}")
    print(f"{'='*80}\n")

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    df_shuffled = df_balanced.sample(frac=1, random_state=seed).reset_index(drop=True)
    train_texts = df_shuffled['text'].values
    train_labels = df_shuffled['label'].values

    train_dataset = ArabicSentimentDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    print(f"🏗️ Création du modèle (seed {seed})...")
    model = SimpleAttentionMARBERT(
        model_name=MODEL_NAME,
        num_labels=3,
        dropout=DROPOUT
    ).to(DEVICE)

    if USE_SWA:
        swa_model = AveragedModel(model)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=0.01,
        betas=(0.9, 0.999),
        eps=1e-8
    )

    total_steps = len(train_loader) * EPOCHS // ACCUMULATION_STEPS
    warmup_steps = int(0.1 * total_steps)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    if USE_SWA:
        swa_scheduler = SWALR(optimizer, swa_lr=SWA_LR)

    # ========================================================================
    # ENTRAÎNEMENT
    # ========================================================================

    print(f"\n🏋️ Entraînement...\n")

    scaler = GradScaler()

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        optimizer.zero_grad()

        for batch_idx, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            with autocast():
                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels)
                loss = loss / ACCUMULATION_STEPS

            scaler.scale(loss).backward()

            if (batch_idx + 1) % ACCUMULATION_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

                if USE_SWA and epoch >= SWA_START_EPOCH:
                    swa_scheduler.step()
                else:
                    scheduler.step()

                optimizer.zero_grad()

            train_loss += loss.item() * ACCUMULATION_STEPS

        if USE_SWA and epoch >= SWA_START_EPOCH:
            swa_model.update_parameters(model)

        avg_train_loss = train_loss / len(train_loader)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"  Epoch {epoch+1}/{EPOCHS} - Loss: {avg_train_loss:.4f} - LR: {current_lr:.6f}")

    print(f"\n✅ Run {run_idx+1} terminé!")

    # ========================================================================
    # PRÉDICTIONS AVEC TTA
    # ========================================================================

    print(f"🔮 Prédictions standard...")

    test_dataset = ArabicSentimentDataset(test_df['text'].values, None, tokenizer, MAX_LENGTH)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    # Regular
    model.eval()

    if USE_TTA:
        print(f"   Avec TTA ({TTA_ROUNDS} rounds)...")
        tta_variations = apply_tta(test_df['text'].values, num_rounds=TTA_ROUNDS)

        all_tta_preds = []
        for variation_texts in tta_variations:
            variation_dataset = ArabicSentimentDataset(variation_texts, None, tokenizer, MAX_LENGTH)
            variation_loader = DataLoader(variation_dataset, batch_size=BATCH_SIZE, shuffle=False)

            var_preds = []
            with torch.no_grad():
                for batch in variation_loader:
                    input_ids = batch['input_ids'].to(DEVICE)
                    attention_mask = batch['attention_mask'].to(DEVICE)

                    with autocast():
                        logits = model(input_ids, attention_mask)

                    probs = F.softmax(logits, dim=1)
                    var_preds.append(probs.cpu().numpy())

            all_tta_preds.append(np.vstack(var_preds))

        run_predictions = np.mean(all_tta_preds, axis=0)
    else:
        test_preds = []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)

                with autocast():
                    logits = model(input_ids, attention_mask)

                probs = F.softmax(logits, dim=1)
                test_preds.append(probs.cpu().numpy())

        run_predictions = np.vstack(test_preds)

    all_predictions.append(run_predictions)

    # SWA predictions
    if USE_SWA:
        print(f"🔮 Prédictions SWA...")

        torch.optim.swa_utils.update_bn(train_loader, swa_model, device=DEVICE)

        swa_model.eval()
        swa_preds = []

        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)

                with autocast():
                    logits = swa_model(input_ids, attention_mask)

                probs = F.softmax(logits, dim=1)
                swa_preds.append(probs.cpu().numpy())

        swa_run_predictions = np.vstack(swa_preds)
        all_swa_predictions.append(swa_run_predictions)

    del model, optimizer, scheduler, train_loader, train_dataset
    if USE_SWA:
        del swa_model, swa_scheduler
    gc.collect()
    torch.cuda.empty_cache()

    print(f"   GPU Memory: {torch.cuda.memory_allocated(DEVICE)/1024**3:.2f}GB")

print(f"\n{'='*80}")
print(f"✅ TOUS LES RUNS TERMINÉS")
print(f"{'='*80}")



🚀 ENTRAÎNEMENT ULTRA-OPTIMISÉ: 7 RUNS

🚀 RUN 1/7 - SEED 42

🏗️ Création du modèle (seed 42)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/654M [00:00<?, ?B/s]

BertModel LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🏋️ Entraînement...

  Epoch 1/15 - Loss: 0.5736 - LR: 0.000012
  Epoch 2/15 - Loss: 0.2735 - LR: 0.000018
  Epoch 3/15 - Loss: 0.1365 - LR: 0.000017
  Epoch 4/15 - Loss: 0.1088 - LR: 0.000017
  Epoch 5/15 - Loss: 0.0934 - LR: 0.000015
  Epoch 6/15 - Loss: 0.0914 - LR: 0.000013
  Epoch 7/15 - Loss: 0.0894 - LR: 0.000012
  Epoch 8/15 - Loss: 0.0889 - LR: 0.000010
  Epoch 9/15 - Loss: 0.0887 - LR: 0.000007
  Epoch 10/15 - Loss: 0.0886 - LR: 0.000005
  Epoch 11/15 - Loss: 0.0885 - LR: 0.000004
  Epoch 12/15 - Loss: 0.0885 - LR: 0.000004
  Epoch 13/15 - Loss: 0.0884 - LR: 0.000004
  Epoch 14/15 - Loss: 0.0891 - LR: 0.000004
  Epoch 15/15 - Loss: 0.0884 - LR: 0.000004

✅ Run 1 terminé!
🔮 Prédictions standard...
   Avec TTA (5 rounds)...
🔮 Prédictions SWA...
   GPU Memory: 3.15GB

🚀 RUN 2/7 - SEED 123

🏗️ Création du modèle (seed 123)...


...


🏋️ Entraînement...

  Epoch 1/15 - Loss: 0.6203 - LR: 0.000012
  Epoch 2/15 - Loss: 0.2780 - LR: 0.000018
  Epoch 3/15 - Loss: 0.1524 - LR: 0.000017
  Epoch 4/15 - Loss: 0.1075 - LR: 0.000017
  Epoch 5/15 - Loss: 0.0967 - LR: 0.000015
  Epoch 6/15 - Loss: 0.0922 - LR: 0.000013
  Epoch 7/15 - Loss: 0.0891 - LR: 0.000012
  Epoch 8/15 - Loss: 0.0898 - LR: 0.000010
  Epoch 9/15 - Loss: 0.0903 - LR: 0.000007
  Epoch 10/15 - Loss: 0.0885 - LR: 0.000005
  Epoch 11/15 - Loss: 0.0885 - LR: 0.000004
  Epoch 12/15 - Loss: 0.0885 - LR: 0.000004
  Epoch 13/15 - Loss: 0.0884 - LR: 0.000004
  Epoch 14/15 - Loss: 0.0884 - LR: 0.000004
  Epoch 15/15 - Loss: 0.0884 - LR: 0.000004

✅ Run 6 terminé!
🔮 Prédictions standard...
   Avec TTA (5 rounds)...
🔮 Prédictions SWA...
   GPU Memory: 3.13GB

🚀 RUN 7/7 - SEED 555

🏗️ Création du modèle (seed 555)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🏋️ Entraînement...

  Epoch 1/15 - Loss: 0.5991 - LR: 0.000012
  Epoch 2/15 - Loss: 0.2660 - LR: 0.000018
  Epoch 3/15 - Loss: 0.1336 - LR: 0.000017
  Epoch 4/15 - Loss: 0.1044 - LR: 0.000017
  Epoch 5/15 - Loss: 0.0986 - LR: 0.000015
  Epoch 6/15 - Loss: 0.0942 - LR: 0.000013
  Epoch 7/15 - Loss: 0.0893 - LR: 0.000012
  Epoch 8/15 - Loss: 0.0898 - LR: 0.000010
  Epoch 9/15 - Loss: 0.0889 - LR: 0.000007
  Epoch 10/15 - Loss: 0.0886 - LR: 0.000005
  Epoch 11/15 - Loss: 0.0885 - LR: 0.000004
  Epoch 12/15 - Loss: 0.0885 - LR: 0.000004
  Epoch 13/15 - Loss: 0.0884 - LR: 0.000004
  Epoch 14/15 - Loss: 0.0884 - LR: 0.000004
  Epoch 15/15 - Loss: 0.0883 - LR: 0.000004

✅ Run 7 terminé!
🔮 Prédictions standard...
   Avec TTA (5 rounds)...
🔮 Prédictions SWA...
   GPU Memory: 3.14GB

✅ TOUS LES RUNS TERMINÉS

🚀 RUN 1/7 - SEED 42



NameError: name 'time' is not defined

In [10]:
# ============================================================================
# ENSEMBLE FINAL + PSEUDO-LABELING (CLEF POUR 84.61%)
# ============================================================================

print(f"\n{'='*80}")
print(f"🎯 ENSEMBLE FINAL")
print(f"{'='*80}\n")

# Combiner
if USE_SWA and len(all_swa_predictions) > 0:
    regular_probs = np.mean(all_predictions, axis=0)
    swa_probs = np.mean(all_swa_predictions, axis=0)
    ensemble_probs = 0.4 * regular_probs + 0.6 * swa_probs
    print("✅ Ensemble: 40% Regular + 60% SWA")
else:
    ensemble_probs = np.mean(all_predictions, axis=0)
    print("✅ Ensemble: Regular only")

# ============================================================================
# PSEUDO-LABELING
# ============================================================================

if USE_PSEUDO_LABELING:
    print(f"\n🔄 Pseudo-Labeling (threshold={PSEUDO_THRESHOLD})...")

    # Identifier échantillons confiants
    confidences = np.max(ensemble_probs, axis=1)
    pseudo_mask = confidences >= PSEUDO_THRESHOLD
    pseudo_indices = np.where(pseudo_mask)[0]

    print(f"   {len(pseudo_indices)}/{len(test_df)} échantillons avec confiance >= {PSEUDO_THRESHOLD}")

    if len(pseudo_indices) > 0:
        # Créer dataset augmenté
        pseudo_labels = np.argmax(ensemble_probs[pseudo_indices], axis=1)
        pseudo_texts = test_df.iloc[pseudo_indices]['text'].values

        # Combiner train + pseudo
        combined_texts = np.concatenate([df_balanced['text'].values, pseudo_texts])
        combined_labels = np.concatenate([df_balanced['label'].values, pseudo_labels])

        print(f"   Nouvel ensemble: {len(combined_texts)} samples ({len(pseudo_indices)} pseudo-labels)")

        # Re-entraîner
        print(f"\n🏋️ Re-entraînement avec pseudo-labels...")

        pseudo_dataset = ArabicSentimentDataset(combined_texts, combined_labels, tokenizer, MAX_LENGTH)
        pseudo_loader = DataLoader(
            pseudo_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=2
        )

        pseudo_model = SimpleAttentionMARBERT(
            model_name=MODEL_NAME,
            num_labels=3,
            dropout=DROPOUT
        ).to(DEVICE)

        pseudo_optimizer = torch.optim.AdamW(
            pseudo_model.parameters(),
            lr=LR * 0.5,
            weight_decay=0.01
        )

        pseudo_scaler = GradScaler()

        # Entraînement rapide (5 epochs)
        for epoch in range(5):
            pseudo_model.train()
            train_loss = 0.0

            for batch in pseudo_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)

                with autocast():
                    logits = pseudo_model(input_ids, attention_mask)
                    loss = criterion(logits, labels)

                pseudo_scaler.scale(loss).backward()
                pseudo_scaler.step(pseudo_optimizer)
                pseudo_scaler.update()
                pseudo_optimizer.zero_grad()

                train_loss += loss.item()

            print(f"  Epoch {epoch+1}/5 - Loss: {train_loss / len(pseudo_loader):.4f}")

        # Prédictions pseudo-model
        print(f"🔮 Prédictions finales avec pseudo-model...")

        test_dataset = ArabicSentimentDataset(test_df['text'].values, None, tokenizer, MAX_LENGTH)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

        pseudo_model.eval()
        pseudo_preds = []

        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)

                with autocast():
                    logits = pseudo_model(input_ids, attention_mask)

                probs = F.softmax(logits, dim=1)
                pseudo_preds.append(probs.cpu().numpy())

        pseudo_probs = np.vstack(pseudo_preds)

        # Combiner ensemble + pseudo-model
        final_probs = 0.7 * ensemble_probs + 0.3 * pseudo_probs
        print("✅ Pseudo-labeling appliqué (70% ensemble + 30% pseudo-model)")

        del pseudo_model, pseudo_optimizer
        gc.collect()
        torch.cuda.empty_cache()
    else:
        final_probs = ensemble_probs
else:
    final_probs = ensemble_probs

# ============================================================================
# PRÉDICTIONS FINALES
# ============================================================================

final_predictions = np.argmax(final_probs, axis=1)

label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
final_predictions_labels = [label_map[pred] for pred in final_predictions]

# Statistiques
confidences = np.max(final_probs, axis=1)

print(f"\n📊 STATISTIQUES FINALES:")
print(f"   - Confiance moyenne: {confidences.mean():.3f}")
print(f"   - Confiance médiane: {np.median(confidences):.3f}")
print(f"   - Confiance min: {confidences.min():.3f}")
print(f"   - Confiance max: {confidences.max():.3f}")

print(f"\n📊 Distribution des prédictions:")
pred_dist = pd.Series(final_predictions_labels).value_counts()
for sentiment, count in pred_dist.items():
    percentage = (count / len(final_predictions_labels)) * 100
    print(f"   {sentiment}: {count} ({percentage:.1f}%)")

# Variance
if len(all_predictions) > 1:
    variance = np.var(all_predictions, axis=0).mean(axis=1)
    print(f"\n📊 Variance entre runs:")
    print(f"   - Variance moyenne: {variance.mean():.4f}")
    print(f"   - Variance médiane: {np.median(variance):.4f}")

# Sauvegarde
submission = pd.DataFrame({
    'ID': range(len(final_predictions_labels)),
    'Sentiment': final_predictions_labels
})

output_file = 'submission_ultra_optimized.csv'
submission.to_csv(output_file, index=False)

print(f"\n✅ Prédictions sauvegardées: {output_file}")
print(f"\n👀 Aperçu:")
print(submission.head(10))

print("\n" + "="*80)
print("🎉 ENTRAÎNEMENT ULTRA-OPTIMISÉ TERMINÉ!")
print("="*80)


🎯 ENSEMBLE FINAL

✅ Ensemble: 40% Regular + 60% SWA

🔄 Pseudo-Labeling (threshold=0.95)...
   0/312 échantillons avec confiance >= 0.95

📊 STATISTIQUES FINALES:
   - Confiance moyenne: 0.875
   - Confiance médiane: 0.932
   - Confiance min: 0.405
   - Confiance max: 0.941

📊 Distribution des prédictions:
   positive: 125 (40.1%)
   negative: 108 (34.6%)
   neutral: 79 (25.3%)

📊 Variance entre runs:
   - Variance moyenne: 0.0103
   - Variance médiane: 0.0000

✅ Prédictions sauvegardées: submission_ultra_optimized.csv

👀 Aperçu:
   ID Sentiment
0   0  positive
1   1  positive
2   2  negative
3   3  positive
4   4  negative
5   5  positive
6   6   neutral
7   7  negative
8   8  positive
9   9  negative

🎉 ENTRAÎNEMENT ULTRA-OPTIMISÉ TERMINÉ!
